# Chapter 5 — Authentication & Login
> *GitHub is a nightclub. You need ID to get in.*
> *Your old password doesn't work anymore. August 2021. Gone. Forever.* 

---

>  The GitHub authentication migration killed many relationships.
> "Why is git push giving me an error?!"
> "Because GitHub removed password auth. Two years ago."
> "...why didn't anyone tell me?"
>
> This chapter is someone telling you.

## Method 1 — HTTPS + Personal Access Token (PAT) 

>  GitHub killed plain password authentication in August 2021.
> Using your actual GitHub password now gives you: `remote: Support for password authentication was removed.`
> Use a PAT (Personal Access Token) instead. It's basically a password with superpowers.

### PAT Classic vs Fine-Grained — Which One?

| Type | Age | Scope | Verdict |
|------|-----|-------|---------|
| **Classic token** | Original | One big permission set covering all your repos |  Simple, still works fine for personal use |
| **Fine-grained token** | 2022 — new default | Per-repo, per-permission control |  More secure — GitHub's recommended choice |

> **Fine-grained = surgical precision.** You say: "This token can ONLY push to `my-project`, ONLY read Issues, nothing else."
> Classic = a master key. Fine-grained = a key that only opens the specific door you choose.

### Create a Fine-Grained PAT (Recommended):
1. GitHub → your avatar (top right) → **Settings**
2. Scroll all the way down → **Developer settings** → **Personal access tokens** → **Fine-grained tokens**
3. Click **Generate new token**
4. Set: name (e.g. "my laptop 2025"), expiry (90 days is sensible), Resource owner = your account
5. **Repository access** → **Only select repositories** → pick your repo(s)
6. **Permissions** → **Repository permissions** → set **Contents: Read and write**
7. Click **Generate token** — **COPY IT NOW. You will never see it again. This is your only chance.** 

### Create a Classic PAT (Simpler):
1. GitHub → Settings → **Developer settings** → **Personal access tokens** → **Tokens (classic)**
2. **Generate new token (classic)** → tick **`repo`** → click Generate → COPY IT

### Use it when Git asks for credentials:
```bash
git push origin main
# Username: your-github-username
# Password: paste-your-PAT-here  ← NOT your actual GitHub password!
```

### Save your PAT so Git stops asking every time:
```powershell
#  Windows — uses Windows Credential Manager (secure, automatic):
git config --global credential.helper manager
```
```bash
#  Mac — uses Keychain:
git config --global credential.helper osxkeychain

#  Linux — caches for 1 hour:
git config --global credential.helper 'cache --timeout=3600'
```

## Method 2 — SSH Keys  (Best for Daily Development)

One-time setup. After that: **zero password prompts. Ever.**
This is what every developer who codes daily eventually switches to.

```bash
# 1. Generate an SSH key pair (one private, one public):
ssh-keygen -t ed25519 -C "your_email@example.com"
# Press Enter to accept default file path (~/.ssh/id_ed25519)
# Set a passphrase for extra security (or leave blank — optional)

# 2. Start SSH agent and register your key:
eval "$(ssh-agent -s)"
ssh-add ~/.ssh/id_ed25519

# 3. Copy your PUBLIC key to share with GitHub:
cat ~/.ssh/id_ed25519.pub          # Print to terminal — then copy manually
cat ~/.ssh/id_ed25519.pub | pbcopy #  Mac — auto-copies to clipboard
cat ~/.ssh/id_ed25519.pub | clip   #  Git Bash — auto-copies to clipboard
Get-Content ~/.ssh/id_ed25519.pub | Set-Clipboard  #  PowerShell

# 4. Add to GitHub:
# Settings → SSH and GPG keys → New SSH key → Paste → Save

# 5. Test it works:
ssh -T git@github.com
# Success looks like: Hi username! You've successfully authenticated!

# 6. Clone + push using SSH URL (instead of HTTPS):
git clone git@github.com:username/repo.git
git remote set-url origin git@github.com:username/repo.git
```

>  SSH keys are like a key card for your apartment building.
> You set it up once. After that you just tap and walk in.
> No typing username + PAT every time like it's 2007.

## Method 3 — GitHub CLI (`gh`)  Easiest Setup in the Universe

```bash
# Install:
winget install --id GitHub.cli    #  Windows
brew install gh                   #  Mac
sudo apt install gh               #  Linux

# Login — it opens a browser, you click a few buttons, done:
gh auth login

# Check if you're logged in:
gh auth status

# Log out:
gh auth logout
```

>  `gh auth login` is genuinely the easiest option for beginners.
> It opens a browser, you click "Authorize", you're done.
> It feels too easy. It IS that easy. Use it.

## Which Method Should You Use?

| Your situation | Use this |
|---------------|----------|
| First day ever, just want it to work | `gh auth login` — 30 seconds, no thinking |
| Occasional personal use | HTTPS + PAT |
| Coding daily, tired of typing credentials | SSH keys — set once, forget forever |
| Corporate / enterprise / team setup | SSH or OAuth |